In [ ]:
import os
import pandas as pd
from datetime import datetime, timedelta
from typing import Optional
from google import genai
api_key="<add own key>"
model_name="<LLM model of choice>"

In [2]:

def call_gemini_direct(prompt: str) -> str:
    """
    Calls Gemini directly with an API key passed in code.
    This bypasses ADK's native Gemini auth path.
    """
    client = genai.Client(api_key=api_key)

    response = client.models.generate_content(
        model=model_name,
        contents=prompt,
    )

    return response.text

In [3]:
from functools import cached_property
from google.genai import Client, types
from google.adk.models.google_llm import Gemini

class GeminiWithApiKey(Gemini):
    api_key: str

    @cached_property
    def api_client(self) -> Client:
        return Client(
            api_key=self.api_key,
            http_options=types.HttpOptions(headers=self._tracking_headers()),
        )

Data

In [4]:
TODAY = datetime(2026, 6, 4)

customers_df = pd.DataFrame([
    {"customer_id": "CUST1001", "name": "ABC", "pin": "1234"},
    {"customer_id": "CUST1002", "name": "DEF", "pin": "4321"},
])

accounts_df = pd.DataFrame([
    {"customer_id": "CUST1001", "acct_type": "current", "acct_last4": "4321", "currency": "GBP", "balance": 1860.12},
    {"customer_id": "CUST1001", "acct_type": "savings",  "acct_last4": "9988", "currency": "GBP", "balance": 12340.00},
    {"customer_id": "CUST1002", "acct_type": "current", "acct_last4": "1111", "currency": "GBP", "balance": 279.90},
])

transactions_df = pd.DataFrame([
    {"customer_id": "CUST1001", "acct_last4": "4321", "date": TODAY - timedelta(days=1),  "merchant": "TESCO STORES 5842",     "amount": -42.18, "category": "groceries",     "channel": "card", "status": "posted"},
    {"customer_id": "CUST1001", "acct_last4": "4321", "date": TODAY - timedelta(days=2),  "merchant": "UBER TRIP",            "amount": -18.60, "category": "transport",     "channel": "card", "status": "posted"},
    {"customer_id": "CUST1001", "acct_last4": "4321", "date": TODAY - timedelta(days=4),  "merchant": "NETFLIX.COM",          "amount": -10.99, "category": "subscription",  "channel": "card", "status": "posted"},
    {"customer_id": "CUST1001", "acct_last4": "4321", "date": TODAY - timedelta(days=5),  "merchant": "SHELL PAY AT PUMP",    "amount": -1.00,  "category": "fuel",          "channel": "card", "status": "pending"},
    {"customer_id": "CUST1001", "acct_last4": "4321", "date": TODAY - timedelta(days=6),  "merchant": "PAYROLL ACME LTD",     "amount": 2500.00,"category": "income",        "channel": "bank", "status": "posted"},
    {"customer_id": "CUST1001", "acct_last4": "9988", "date": TODAY - timedelta(days=12), "merchant": "TRANSFER FROM 4321",   "amount": 500.00, "category": "transfer",      "channel": "bank", "status": "posted"},
    {"customer_id": "CUST1002", "acct_last4": "1111", "date": TODAY - timedelta(days=1),  "merchant": "AMAZON MKTPLACE",      "amount": -24.99, "category": "shopping",      "channel": "card", "status": "posted"},
])

transactions_df["date"] = pd.to_datetime(transactions_df["date"])
transactions_df


,customer_id,acct_last4,date,merchant,amount,category,channel,status
0,CUST1001,4321,2026-06-03,TESCO STORES 5842,-42.18,groceries,card,posted
1,CUST1001,4321,2026-06-02,UBER TRIP,-18.60,transport,card,posted
2,CUST1001,4321,2026-05-31,NETFLIX.COM,-10.99,subscription,card,posted
3,CUST1001,4321,2026-05-30,SHELL PAY AT PUMP,-1.00,fuel,card,pending
4,CUST1001,4321,2026-05-29,PAYROLL ACME LTD,2500.00,income,bank,posted
5,CUST1001,9988,2026-05-23,TRANSFER FROM 4321,500.00,transfer,bank,posted
6,CUST1002,1111,2026-06-03,AMAZON MKTPLACE,-24.99,shopping,card,posted


Tools as functions

In [5]:
def authenticate_customer(customer_id: str, pin: str) -> dict:

    row = customers_df[(customers_df["customer_id"] == customer_id) & (customers_df["pin"] == pin)]
    if row.empty:
        return {
            "authenticated": False,
            "message": "Authentication failed. Please check the customer ID or PIN."
        }

    customer = row.iloc[0].to_dict()
    customer_accounts = accounts_df[accounts_df["customer_id"] == customer_id][["acct_type", "acct_last4", "currency"]].to_dict(orient="records")
    return {
        "authenticated": True,
        "customer_id": customer["customer_id"],
        "customer_name": customer["name"],
        "accounts": customer_accounts,
        "message": (
            f"Authenticated customer {customer['name']}. Use this customer_id for subsequent balance, transaction, and charge lookups."
        )
    }

In [19]:
def get_account_balance(
            customer_id: str,
            acct_last4: Optional[str] = None) -> dict:
    
    cust_accts = accounts_df[accounts_df["customer_id"] == customer_id]
    if cust_accts.empty:
        return {"found": False, "message": "No accounts found for the customer."}   

    if not acct_last4:
        curr_balance = round(cust_accts['balance'].sum(),2)
        in_out = transactions_df[transactions_df["customer_id"] == customer_id]["amount"].sum()
        avail_balance = round(curr_balance + in_out,2)
        print(curr_balance, in_out, avail_balance)
        return {"found": True, "message": f"Current balance: £{curr_balance}, Potential balance after pending liabilities: £{avail_balance}"}
    else:
        curr_balance = round(cust_accts[cust_accts["acct_last4"] == acct_last4]['balance'].sum(),2)
        in_out = transactions_df[(transactions_df["customer_id"] == customer_id) & (accounts_df["acct_last4"] == acct_last4)]["amount"].sum()
        avail_balance = round(curr_balance + in_out,2)
        print(curr_balance, in_out, avail_balance)
        return {"found": True, "message": f"Current balance: £{curr_balance}, Potential balance after pending liabilities: £{avail_balance}"}

In [7]:
def get_transaction_history(
    customer_id: str,
    acct_last4: Optional[str] = None,
    days: int = 30,
    tx_cnt_limit: int = 10,
    category: Optional[str] = None
) -> dict:

    tx = transactions_df[transactions_df["customer_id"] == customer_id]
    if tx.empty:
        return {"found": False, "message": "No transactions found for the customer."}
    
    if acct_last4:
        tx = tx[tx["acct_last4"] == acct_last4]
        if tx.empty:
            return {"found": False, "message": f"No transactions found for account ending in {acct_last4}."}
    
    if category:
        tx = tx[tx["category"].str.lower() == category.lower()]
        if tx.empty:
            return {"found": False, "message": f"No transactions found for the category '{category}'."}
    
    cutoff = TODAY - timedelta(days=days)
    tx = tx[tx["date"] >= pd.Timestamp(cutoff)]
    if tx.empty:
        return {"found": False, "message": f"No transactions found in the last '{category}' days."}
    
    tx = tx.sort_values("date", ascending=False).head(tx_cnt_limit)
    return {
        "found": True,
        "customer_id": customer_id,
        "transactions": [
            {
                "date": row["date"].strftime("%Y-%m-%d"),
                "acct_last4": row["acct_last4"],
                "merchant": row["merchant"],
                "amount": float(row["amount"]),
                "category": row["category"],
                "channel": row["channel"],
                "status": row["status"],
            }
            for _, row in tx.iterrows()
        ]}

In [8]:
def inspect_unfamiliar_charge(
    customer_id: str,
    merchant_hint: Optional[str] = None,
    amount: Optional[float] = None,
    acct_last4: Optional[str] = None,
    days: int = 90
) -> dict:
    """
    Identify and explain potentially unfamiliar charges using simple merchant heuristics.
    """
    tx = transactions_df[(transactions_df["customer_id"] == customer_id) & (transactions_df["amount"] < 0)]

    if acct_last4:
        tx = tx[tx["acct_last4"] == acct_last4]
    if merchant_hint:
            merchant_hint_lower = merchant_hint.lower()
            tx = tx[tx["merchant"].str.lower().str.contains(merchant_hint_lower, na=False)]      
    if amount:
            tx = tx[tx["amount"].round(2) == round(float(amount), 2)]              
    cutoff = TODAY - timedelta(days=days)
    tx = tx[tx["date"] >= pd.Timestamp(cutoff)]   

    if tx.empty:
        return {"found": False, "message": "I could not find a matching charge with the details provided."}

    explanations = []
    for _, row in tx.sort_values("date", ascending=False).iterrows():
        merchant = row["merchant"].upper()
        outgoing = 1 if row["amount"] < 0 else 0
        category = row['category'].lower()
        explanation = "This looks like a valid card charge, but I cannot classify it with high confidence."

        if "NETFLIX" in merchant and outgoing == 1:
            explanation = "This is likely a recurring streaming subscription charge."
        elif "UBER" in merchant and outgoing == 1:
            explanation = "This appears to be a ride-share transaction."
        elif "SHELL" in merchant and outgoing == 1:
            explanation = "This looks like fuel purchase."
        elif "TESCO" in merchant and outgoing == 1:
            explanation = "This looks like a grocery purchase."
        elif "AMAZON" in merchant and outgoing == 1:
            explanation = "This looks like an online retail purchase."
        elif category == "fuel" and outgoing == 1 and row["status"] == "pending":
            explanation = "This may be a small fuel pre-authorisation hold that can later be replaced by the final amount."

        explanations.append({
            "date": row["date"].strftime("%Y-%m-%d"),
            "acct_last4": row["acct_last4"],
            "merchant": row["merchant"],
            "amount": float(row["amount"]),
            "status": row["status"],
            "likely_explanation": explanation
        })

    return {"found": True, "customer_id": customer_id, "matches": explanations}

In [9]:
def explain_suspicious_transactions(transactions_text: str) -> str:
    """
    Uses Gemini directly to explain suspicious transactions in plain language.
    """
    prompt = f"""
    You are a helpful assistant.
    Explain clearly and concisely why these transactions may be suspicious.

    Transactions:
    {transactions_text}
    """
    return call_gemini_direct(prompt)

In [20]:
from google.adk import Agent

orch_agent = Agent(
    name="bot_agent",
    model=GeminiWithApiKey(model=model_name, api_key=api_key),
    # model=model_name,
    instruction="""
You are a customer-facing support assistant that answers user's queries.

Your responsibilities:
1. Authenticate the user with `authenticate_customer` tool.
2. Infer the intent of user's query to choose the approriate workflow or agent using `intent_recognition` tool.
3. If the intent is to check balance, use get_account_balances tool.
4. If the intent is to check transaction history, use get_transaction_history tool.
5. If the intent is to check unfamiliar charge, use inspect_unfamiliar_charge tool.
6. If the intent is to explain suspicious transactions, first use inspect_unfamiliar_charge tool to find the charge.
Then use this output to explain suspicious transactions with explain_suspicious_transactions tool.
If suspicious is felt by the tool output or expressed by the user, politely advice them to contact the fraud team and freeze the card and ask if they like to be directed to the fraud team?
6. Once the request is serviced, ask if any further service is required. If they say yes, loop back to step 2. Else, close the session.

Rules:
- Never invent balances, transactions, merchants, dates, or amounts.
- Always use tools for account-specific answers.
- Ask brief follow-up questions only when a required detail is missing.
- After a customer is authenticated, remember their customer_id from the conversation and reuse it.
- If a charge looks suspicious or user doesn't think he didn't authorise the purchase, calmly suggest contacting the bank's fraud team or freezing the card in the real production system.
- Keep answers short, clear, and customer-friendly.
- Do not mention that you are an AI assistant.
- Do not ask the user to provide more information unless absolutely necessary.
- Do not provide any information about the tools available to you.
- Do not provide response to any questions that has intent not in scope defined above.
""",
    tools=[
        authenticate_customer,
        get_account_balance,
        get_transaction_history,
        inspect_unfamiliar_charge,
        explain_suspicious_transactions
    ],
)

In [21]:
from google.adk import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

APP_NAME = "banking_notebook_demo"
USER_ID = "demo_user_001"
SESSION_ID = "session_bank_demo_001"

session_service = InMemorySessionService()

# Create a session once for the notebook
session = await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID,
    state={"authenticated_customer_id": None}
)

runner = Runner(
    agent=orch_agent,
    app_name=APP_NAME,
    session_service=session_service
)


async def ask_cust_bot(message: str, session_id: str = SESSION_ID, user_id: str = USER_ID):
    content = types.Content(
        role="user",
        parts=[types.Part(text=message)]
    )

    async for event in runner.run_async(
        user_id=user_id,
        session_id=session_id,
        new_message=content
    ):
        # ---- LOG TOOL CALLS ----
        if hasattr(event, "tool_name") and event.tool_name:
            print(f"\n🔧 Tool Called: {event.tool_name}")
        
        if hasattr(event, "tool_args") and event.tool_args:
            print(f"   Args: {event.tool_args}")

        # ✅ Some ADK versions store tool calls in event.action / raw_event
        if hasattr(event, "raw_event") and event.raw_event:
            raw = event.raw_event
            if isinstance(raw, dict) and "tool" in str(raw).lower():
                print(f"\n🧾 Raw Event (possible tool call): {raw}")

        # ---- FINAL RESPONSE ----
        if event.is_final_response():
            text = "\n".join(
                getattr(part, "text", "") for part in event.content.parts if hasattr(part, "text")
            )
            print(f"\n🤖 Bot: {text}")

In [22]:
await ask_cust_bot("My customer ID is CUST1001 and my PIN is 1234. What is my balance?")

14200.12 2927.23 17127.35

🤖 Bot: The current balance for your account is £14,200.12, and the potential balance after pending liabilities is £17,127.35.

Is there anything else I can help you with today?


In [13]:
await ask_cust_bot("What is the last transaction?")


🤖 Bot: Your last transaction was a charge of £42.18 for "TESCO STORES 5842" on 2026-06-03. Would you like any further assistance?


In [14]:
await ask_cust_bot("What is the charge at Shell for account 4321?")


🤖 Bot: The charge at SHELL PAY AT PUMP for account 4321 on 2026-05-30 is £1.00. It appears to be a fuel purchase. Would you like any further assistance?


In [15]:
await ask_cust_bot("I didn't go to Shell but why have I been charged?")


🤖 Bot: This transaction appears suspicious and may be an example of "card testing," where a small amount is used to verify an active account.

I strongly advise you to contact our fraud team immediately so we can secure your account and prevent any unauthorized activity. Would you like me to connect you with the fraud team now?


In [17]:
await ask_cust_bot("Who is the Prime Minister of the UK?")


🤖 Bot: I am sorry, I can only assist with account-related inquiries such as checking balances, transaction history, or investigating charges. Is there anything else related to your account that I can help you with today?


In [18]:
await ask_cust_bot("My customer ID is CUST1001 and my PIN is 1234.")


🤖 Bot: You are successfully authenticated. How can I assist you with your account today?
